# ACTIVIDAD DE CLASIFICACIÓN DE TEXTO

En esta actividad vamos a trabajar en clasificar textos. Se recorrerá todo el proceso desde traer el dataset hasta proceder a dicha clasificación. Durante la actividad se llevarán a cabo muchos procesos como la creación de un vocabulario, el uso de embeddings y la creación de modelos.

Las cuestiones presentes en esta actividad están basadas en un Notebook creado por François Chollet, uno de los creadores de Keras y autor del libro "Deep Learning with Python".

En este Notebook se trabaja con el dataset "Newsgroup20" que contiene aproximadamente 20000 mensajes que pertenecen a 20 categorías diferentes.

El objetivo es entender los conceptos que se trabajan y ser capaz de hacer pequeñas experimentaciones para mejorar el Notebook creado.

# Librerías

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
import pathlib
from tensorflow import keras

data_path = keras.utils.get_file(
    "news20.tar.gz",
    "http://www.cs.cmu.edu/afs/cs.cmu.edu/project/theo-20/www/data/news20.tar.gz",
    untar=True,
)

print("data_path:", data_path)
data_dir = pathlib.Path(data_path).parent / "20_newsgroup"
print("data_dir:", data_dir)
print("¿Existe?:", data_dir.exists())

# Ver qué hay en la carpeta padre
parent = pathlib.Path(data_path).parent
print("\nContenido de la carpeta padre:")
print(os.listdir(parent))

data_path: /root/.keras/datasets/news20_extracted
data_dir: /root/.keras/datasets/20_newsgroup
¿Existe?: False

Contenido de la carpeta padre:
['news20.tar.gz', 'news20_extracted']


In [ ]:
import os
import pathlib

# La carpeta real es news20_extracted
extracted_dir = pathlib.Path("/root/.keras/datasets/news20_extracted")

print("Contenido de news20_extracted:")
print(os.listdir(extracted_dir))

Contenido de news20_extracted:
['20_newsgroup']


In [ ]:
import spacy
import os
import pathlib

# Cargar modelo de spaCy
nlp = spacy.load("en_core_web_sm")

# Ruta correcta
data_dir = pathlib.Path("/root/.keras/datasets/news20_extracted/20_newsgroup")
graphics_dir = data_dir / "comp.graphics"

print("¿Existe comp.graphics?:", graphics_dir.exists())

# Tomar muestra de 15 archivos
fnames = os.listdir(graphics_dir)[:15]

# Calcular tokens por archivo
token_counts = []
for fname in fnames:
    fpath = graphics_dir / fname
    with open(fpath, encoding="latin-1") as f:
        content = f.read()
    doc = nlp(content)
    token_counts.append(len(doc))
    print(f"{fname}: {len(doc)} tokens")

# Promedio
promedio = sum(token_counts) / len(token_counts)
print(f"\nPromedio de tokens en 15 archivos de comp.graphics: {promedio:.2f}")

¿Existe comp.graphics?: True
39027: 157 tokens
38840: 74 tokens
38421: 157 tokens
38258: 582 tokens
38377: 13965 tokens
38573: 321 tokens
39638: 13075 tokens
37957: 159 tokens
38277: 416 tokens
38426: 429 tokens
38988: 139 tokens
39625: 195 tokens
39629: 468 tokens
38396: 161 tokens
38899: 209 tokens

Promedio de tokens en 15 archivos de comp.graphics: 2033.80


In [ ]:
data_dir = pathlib.Path("/root/.keras/datasets/news20_extracted/20_newsgroup")

# Abrir un archivo y ver las primeras líneas
fpath = data_dir / "comp.graphics" / "37261"
with open(fpath, encoding="latin-1") as f:
    content = f.read()

lines = content.split("\n")

print("=== PRIMERAS 15 LÍNEAS (encabezado) ===")
for i, line in enumerate(lines[:15]):
    print(f"Línea {i}: {line}")

print("\n=== CONTENIDO DESPUÉS DE DESCARTAR 10 LÍNEAS ===")
print("\n".join(lines[10:15]))

=== PRIMERAS 15 LÍNEAS (encabezado) ===
Línea 0: Xref: cantaloupe.srv.cs.cmu.edu comp.graphics:37261 alt.graphics:519 comp.graphics.animation:2614
Línea 1: Path: cantaloupe.srv.cs.cmu.edu!das-news.harvard.edu!ogicse!uwm.edu!zaphod.mps.ohio-state.edu!darwin.sura.net!dtix.dt.navy.mil!oasys!lipman
Línea 2: From: lipman@oasys.dt.navy.mil (Robert Lipman)
Línea 3: Newsgroups: comp.graphics,alt.graphics,comp.graphics.animation
Línea 4: Subject: CALL FOR PRESENTATIONS: Navy SciViz/VR Seminar
Línea 5: Message-ID: <32850@oasys.dt.navy.mil>
Línea 6: Date: 19 Mar 93 20:10:23 GMT
Línea 7: Article-I.D.: oasys.32850
Línea 8: Expires: 30 Apr 93 04:00:00 GMT
Línea 9: Reply-To: lipman@oasys.dt.navy.mil (Robert Lipman)
Línea 10: Followup-To: comp.graphics
Línea 11: Distribution: usa
Línea 12: Organization: Carderock Division, NSWC, Bethesda, MD
Línea 13: Lines: 65
Línea 14: 

=== CONTENIDO DESPUÉS DE DESCARTAR 10 LÍNEAS ===
Followup-To: comp.graphics
Distribution: usa
Organization: Carderock Division, NS

In [18]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

# Descarga de Datos

In [19]:
data_path = keras.utils.get_file(
    "news20.tar.gz",
    "http://www.cs.cmu.edu/afs/cs.cmu.edu/project/theo-20/www/data/news20.tar.gz",
    untar=True,
)

In [20]:
import os
import pathlib

#Estructura de directorios del dataset
data_dir = pathlib.Path("/root/.keras/datasets/news20_extracted/20_newsgroup")#pathlib.Path(data_path).parent / "20_newsgroup"
dirnames = os.listdir(data_dir)
print("Number of directories:", len(dirnames))
print("Directory names:", dirnames)

Number of directories: 20
Directory names: ['sci.med', 'comp.sys.mac.hardware', 'comp.windows.x', 'rec.autos', 'comp.os.ms-windows.misc', 'talk.religion.misc', 'sci.space', 'comp.sys.ibm.pc.hardware', 'alt.atheism', 'talk.politics.guns', 'misc.forsale', 'sci.crypt', 'comp.graphics', 'rec.motorcycles', 'soc.religion.christian', 'talk.politics.mideast', 'rec.sport.baseball', 'sci.electronics', 'talk.politics.misc', 'rec.sport.hockey']


In [21]:
print(data_dir)

/root/.keras/datasets/news20_extracted/20_newsgroup


In [22]:
#Algunos archivos de la categoria "com.graphics"
fnames = os.listdir(data_dir / "comp.graphics")
print("Number of files in comp.graphics:", len(fnames))
print("Some example filenames:", fnames[:5])

Number of files in comp.graphics: 1000
Some example filenames: ['38874', '38545', '39736', '37932', '38760']


In [23]:
#Ejemplo de un texto de la categoría "com.graphics"
print(open(data_dir / "comp.graphics" / "37261").read())

Xref: cantaloupe.srv.cs.cmu.edu comp.graphics:37261 alt.graphics:519 comp.graphics.animation:2614
Path: cantaloupe.srv.cs.cmu.edu!das-news.harvard.edu!ogicse!uwm.edu!zaphod.mps.ohio-state.edu!darwin.sura.net!dtix.dt.navy.mil!oasys!lipman
From: lipman@oasys.dt.navy.mil (Robert Lipman)
Newsgroups: comp.graphics,alt.graphics,comp.graphics.animation
Subject: CALL FOR PRESENTATIONS: Navy SciViz/VR Seminar
Message-ID: <32850@oasys.dt.navy.mil>
Date: 19 Mar 93 20:10:23 GMT
Article-I.D.: oasys.32850
Expires: 30 Apr 93 04:00:00 GMT
Reply-To: lipman@oasys.dt.navy.mil (Robert Lipman)
Followup-To: comp.graphics
Distribution: usa
Organization: Carderock Division, NSWC, Bethesda, MD
Lines: 65


			CALL FOR PRESENTATIONS
	
      NAVY SCIENTIFIC VISUALIZATION AND VIRTUAL REALITY SEMINAR

			Tuesday, June 22, 1993

	    Carderock Division, Naval Surface Warfare Center
	      (formerly the David Taylor Research Center)
			  Bethesda, Maryland

SPONSOR: NESS (Navy Engineering Software System) is sponsori

In [24]:
#Algunos archivos de la categoria "talk.politics.misc"
fnames = os.listdir(data_dir / "talk.politics.misc")
print("Number of files in talk.politics.misc:", len(fnames))
print("Some example filenames:", fnames[:5])

Number of files in talk.politics.misc: 1000
Some example filenames: ['178883', '178706', '178366', '178681', '178961']


In [25]:
#Ejemplo de un texto de la categoría "talk.politics.misc"
print(open(data_dir / "talk.politics.misc" / "178463").read())

Xref: cantaloupe.srv.cs.cmu.edu talk.politics.guns:54219 talk.politics.misc:178463
Newsgroups: talk.politics.guns,talk.politics.misc
Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!magnus.acs.ohio-state.edu!usenet.ins.cwru.edu!agate!spool.mu.edu!darwin.sura.net!martha.utcc.utk.edu!FRANKENSTEIN.CE.UTK.EDU!VEAL
From: VEAL@utkvm1.utk.edu (David Veal)
Subject: Re: Proof of the Viability of Gun Control
Message-ID: <VEAL.749.735192116@utkvm1.utk.edu>
Lines: 21
Sender: usenet@martha.utcc.utk.edu (USENET News System)
Organization: University of Tennessee Division of Continuing Education
References: <1qpbqd$ntl@access.digex.net> <C5otvp.ItL@magpie.linknet.com>
Date: Mon, 19 Apr 1993 04:01:56 GMT

[alt.drugs and alt.conspiracy removed from newsgroups line.]

In article <C5otvp.ItL@magpie.linknet.com> neal@magpie.linknet.com (Neal) writes:

>   Once the National Guard has been called into federal service,
>it is under the command of the present. Tha N

In [26]:
list_all_dir = [
    'alt.atheism',
    'comp.graphics',
    'comp.sys.mac.hardware',
    'comp.windows.x',
    'misc.forsale',
    'rec.autos',
    'rec.sport.baseball',
    'rec.sport.hockey',
    'sci.crypt',
    'sci.med',
    'sci.space',
    'soc.religion.christian',
    'talk.politics.guns',
    'talk.politics.misc',
    'talk.religion.misc'
]

In [27]:
samples = []
labels = []
class_names = []
class_index = 0
for dirname in list_all_dir:
    class_names.append(dirname)
    dirpath = data_dir / dirname
    fnames = os.listdir(dirpath)
    print("Processing %s, %d files found" % (dirname, len(fnames)))
    for fname in fnames:
        fpath = dirpath / fname
        f = open(fpath, encoding="latin-1")
        content = f.read()
        lines = content.split("\n")
        lines = lines[10:]
        content = "\n".join(lines)
        samples.append(content)
        labels.append(class_index)
    class_index += 1

print("Classes:", class_names)
print("Number of samples:", len(samples))

Processing alt.atheism, 1000 files found
Processing comp.graphics, 1000 files found
Processing comp.sys.mac.hardware, 1000 files found
Processing comp.windows.x, 1000 files found
Processing misc.forsale, 1000 files found
Processing rec.autos, 1000 files found
Processing rec.sport.baseball, 1000 files found
Processing rec.sport.hockey, 1000 files found
Processing sci.crypt, 1000 files found
Processing sci.med, 1000 files found
Processing sci.space, 1000 files found
Processing soc.religion.christian, 997 files found
Processing talk.politics.guns, 1000 files found
Processing talk.politics.misc, 1000 files found
Processing talk.religion.misc, 1000 files found
Classes: ['alt.atheism', 'comp.graphics', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.misc', 'talk.religion.misc']
Number of samples: 14997


# Mezclando los datos para separarlos en Traning y Test

In [28]:
# Shuffle the data
seed = 1337
rng = np.random.RandomState(seed)
rng.shuffle(samples)
rng = np.random.RandomState(seed)
rng.shuffle(labels)
keras.utils.set_random_seed(seed)

# Extract a training & validation split
validation_split = 0.2
num_validation_samples = int(validation_split * len(samples))
train_samples = samples[:-num_validation_samples]
val_samples = samples[-num_validation_samples:]
train_labels = labels[:-num_validation_samples]
val_labels = labels[-num_validation_samples:]

In [13]:
print("=== Ejemplo de train_samples ===")
print(train_samples[0])

print("\n=== Ejemplo de val_samples ===")
print(val_samples[0])

print("\n=== Ejemplo de train_labels ===")
print(train_labels[0])

print("\n=== Ejemplo de val_labels ===")
print(val_labels[0])

=== Ejemplo de train_samples ===
References: <13APR199308003715@delphi.gsfc.nasa.gov>  <1993Apr15.190711.22190@walter.bellcore.com>,<1993Apr15.173902.66278@cc.usu.edu>
Reply-To: carl@SOL1.GPS.CALTECH.EDU
NNTP-Posting-Host: sol1.gps.caltech.edu

In article <1993Apr15.173902.66278@cc.usu.edu>, slyx0@cc.usu.edu writes:
=Surprise surprise, different people react differently to different things. One
=slightly off the subject case in point. My brother got stung by a bee. I know
=he is allergic to bee stings, but that his reaction is severe localized
=swelling, not anaphylactic shock. I could not convince the doctors of that,
=however, because that's not written in their little rule book.

Of course, bee venom isn't a single chemical.  Could be your brother is
reacting to a different component than the one that causes anaphylactic shock
in other people.

Similarly, Chinese food isn't just MSG.  There are a lot of other ingredients
in it.  Why, when someone eats something with lots of ingredie

# Tokenización de las palabras con TextVectorization

In [29]:
from tensorflow.keras.layers import TextVectorization
vectorizer = TextVectorization(max_tokens=20000, output_sequence_length=200)
text_ds = tf.data.Dataset.from_tensor_slices(train_samples).batch(128)
vectorizer.adapt(text_ds)

In [30]:
vectorizer.get_vocabulary()[:5]

['', '[UNK]', np.str_('the'), np.str_('to'), np.str_('of')]

In [31]:
len(vectorizer.get_vocabulary())

20000

In [32]:
# Mensaje corto (menos de 200 palabras)
mensaje_corto = ["hola mundo"]

# Mensaje largo (más de 200 palabras)
mensaje_largo = ["word " * 300]  # 300 palabras repetidas

salida_corta = vectorizer([mensaje_corto])
salida_larga = vectorizer([mensaje_largo])

print("=== Mensaje corto ===")
print("Longitud de salida:", salida_corta.shape)
print("Últimos 10 valores (deben ser ceros si es corto):", salida_corta.numpy()[0, -10:])

print("\n=== Mensaje largo ===")
print("Longitud de salida:", salida_larga.shape)
print("Primeros 10 valores:", salida_larga.numpy()[0, :10])
print("Últimos 10 valores (debe estar recortado en 200):", salida_larga.numpy()[0, -10:])

=== Mensaje corto ===
Longitud de salida: (1, 200)
Últimos 10 valores (deben ser ceros si es corto): [0 0 0 0 0 0 0 0 0 0]

=== Mensaje largo ===
Longitud de salida: (1, 200)
Primeros 10 valores: [340 340 340 340 340 340 340 340 340 340]
Últimos 10 valores (debe estar recortado en 200): [340 340 340 340 340 340 340 340 340 340]


# Viendo la salida de Vectorizer

In [33]:
output = vectorizer([["the cat sat on the mat"]])
output.numpy()[0, :6]

array([   2, 3992, 2115,   18,    2, 6771])

In [34]:
output

<tf.Tensor: shape=(1, 200), dtype=int64, numpy=
array([[   2, 3992, 2115,   18,    2, 6771,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,   

In [35]:
voc = vectorizer.get_vocabulary()
word_index = dict(zip(voc, range(len(voc))))

In [36]:
test = ["the", "cat", "sat", "on", "the", "mat"]
[word_index[w] for w in test]

[2, 3992, 2115, 18, 2, 6771]

# Tokenización de los datos de entrenamiento y validación

In [37]:
x_train = vectorizer(np.array([[s] for s in train_samples])).numpy()
x_val = vectorizer(np.array([[s] for s in val_samples])).numpy()

y_train = np.array(train_labels)
y_val = np.array(val_labels)

# Creación y entrenamiento del modelo. Red Neuronal Clásica

In [38]:
modeloClasico = keras.models.Sequential()
modeloClasico.add(keras.layers.Embedding(20000, 10))
modeloClasico.add(keras.layers.Flatten())
modeloClasico.add(keras.layers.Dense(512, activation='relu'))
modeloClasico.add(keras.layers.Dropout(0.3))
modeloClasico.add(keras.layers.Dense(20, activation='softmax'))

In [39]:
modeloClasico.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modeloClasico.compile(loss="sparse_categorical_crossentropy", optimizer="rmsprop", metrics=["acc"])
modeloClasico.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_val, y_val))
print(modeloClasico.summary())

Epoch 1/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - acc: 0.1279 - loss: 2.6601 - val_acc: 0.2074 - val_loss: 2.4227
Epoch 2/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - acc: 0.3039 - loss: 2.1127 - val_acc: 0.3351 - val_loss: 1.9299
Epoch 3/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - acc: 0.5236 - loss: 1.4948 - val_acc: 0.4678 - val_loss: 1.5148
Epoch 4/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - acc: 0.6936 - loss: 1.0027 - val_acc: 0.5622 - val_loss: 1.2453
Epoch 5/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - acc: 0.8049 - loss: 0.6643 - val_acc: 0.6155 - val_loss: 1.1075
Epoch 6/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - acc: 0.8691 - loss: 0.4552 - val_acc: 0.6369 - val_loss: 1.0630
Epoch 7/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - acc: 0.9051 - loss: 0.3297 - val_acc: 0.6492 - val_loss: 1.0414
Epoch 8/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - acc: 0.9247 - loss: 0.2531 - val_acc: 0.6516 - val_loss: 1.0811
Epoch 9/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - acc: 0.9357

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 10)        │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,024,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │        10,260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,469,546 (9.42 MB)

 Trainable params: 1,234,772 (4.71 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,234,774 (4.71 MB)

None


# Evaluación

In [40]:
string_input = keras.Input(shape=(1,), dtype="string")
x = vectorizer(string_input)
preds = modeloClasico(x)
end_to_end_model = keras.Model(string_input, preds)

In [41]:
probabilities = end_to_end_model(
    keras.ops.convert_to_tensor(
        [["this message is about computer graphics and 3D modeling"]]
    )
)

print(class_names[np.argmax(probabilities[0])])

comp.graphics


In [42]:
probabilities = end_to_end_model(
    keras.ops.convert_to_tensor(
        [["politics and federal courts law that people understand with politician and elects congressman"]]
    )
)

print(class_names[np.argmax(probabilities[0])])

comp.windows.x


In [43]:
probabilities = end_to_end_model(
    keras.ops.convert_to_tensor(
        [["we are talking about religion"]]
    )
)

print(class_names[np.argmax(probabilities[0])])

comp.windows.x
